In [1]:
import numpy as np
from scipy.io.netcdf import netcdf_file
from scipy.interpolate import griddata
from datetime import datetime
from os.path import join
from MyLib import GPSConverter
from pandas import to_datetime

In [2]:
# mitgcm_grid_file = "E:/ALPLakes/Codes/MITgcm_input_output/input_generator/LemanGrid_HR50.nc"
mitgcm_grid_file = "G:/ALPLakes/D3D/lac_de_joux/LDJ_10days_2020/mitgcm_format/joux_grid.nc"
# mitgcm_grid_file = "E:/ALPLakes/Codes/MITgcm_input_output/input_generator/grid_mitgcm_net_roms.nc"
out_fmt = "float"

name = ("uwind", "vwind", "wspeed")

In [3]:
# extract COSMO grid info
def get_MITgcm_grid(mitgcm_file):
    #%Extract geographical data from the first COSMO-2 file
    #date = datevec(dateini); %Date vector
    #FileName = [datapath sprintf('%i',date(1)) '\cosmo2_epfl_lakes_' sprintf('%i%02i%02i',date(1:3)) '.nc'];
    f = netcdf_file(mitgcm_file, 'r')
    xC = f.variables["XC"][:].copy().T
    yC = f.variables["YC"][:].copy().T
    aC = f.variables["AngleCS"][:].copy().T
    aS = f.variables["AngleSN"][:].copy().T
    f.close()
    return (xC, yC), (aC, aS)

def rotate(u, v, cs, sn):
    # rotation of the reference frame
    return (cs*u + sn*v), (-sn*u + cs*v)

# extract COSMO grid info
def get_COSMO_grid(cosmo_file):
    #%Extract geographical data from the first COSMO-2 file
    #date = datevec(dateini); %Date vector
    #FileName = [datapath sprintf('%i',date(1)) '\cosmo2_epfl_lakes_' sprintf('%i%02i%02i',date(1:3)) '.nc'];
    f = netcdf_file(cosmo_file, 'r')
    lon = f.variables["lon_1"][:].copy()
    lat = f.variables["lat_1"][:].copy()
    f.close()
    converter = GPSConverter()
    x = np.zeros_like(lon)
    y = np.zeros_like(lat)
    for jj in range(x.shape[1]):
        for ii in range(x.shape[0]):
            x[ii,jj], y[ii,jj], _ = converter.WGS84toLV03(
                                    lat[ii, jj], lon[ii,jj], 0.0)
    return x, y

# get time axis info from COSMO file
def get_time(ncf):
    time = ncf.variables["time"][:].copy().astype("int")
    refdate = ncf.variables["time"].units.decode('utf8') # modifed by Fazel
    refdate = refdate.split()
    if refdate[0] == "seconds":
        units = "[s]"
    elif refdate[0] == "hours":
        units = "[h]"
    else:
        raise ValueError("Only seconds are implemented")
    refdate = refdate[2].rstrip(",; ") + " " + refdate[3].rstrip(",; ")
    
    return np.array([np.datetime64(refdate) + np.timedelta64(t, units)
                    for t in time])

def cosmo2delft3d_uv(data_path, data_root, start, end,
                  xinfo, yinfo,output_path, spin_up=0): # modified by Fazel

    spin_up = int(spin_up)

    # Time range to be extracted
    date_start = np.datetime64(start)
    date_end = np.datetime64(end)

    # Get grid info
    fname = data_root + to_datetime(date_start).strftime("%Y%m%d") + ".nc"
    fname = join(data_path,fname)

    # generate output grid
    (xM, yM), (cM, sM) = get_MITgcm_grid(mitgcm_grid_file)

    # open the output files
    fls = {}
    for e in name:
        out_name = output_path+"%s_%s_%s.bin" % \
                   (e, start.replace('-',''), end.replace('-',''))
        fls[e] = open(out_name, "ab")

    # loop over COSMO-2 data
    today = date_start
    missing_day = False
    while today<=date_end:
#         # This is to follow a change in the naming convention
#         # so stupid one cannot believe
#         if today <= np.datetime64("2015-05-10"):
#             date_file = today
#         else:
#             date_file = today + np.timedelta64(1, 'D')

        date_file = today
        fname = data_root + to_datetime(date_file).strftime("%Y%m%d") + ".nc"

        fname = join(data_path, fname)
        try:
            xc, yc = get_COSMO_grid(fname)
            ind = np.where((xc>=xinfo[0]-15e3) & (xc<=xinfo[1]+15e3) & 
                           (yc>=yinfo[0]-15e3) & (yc<=yinfo[1]+15e3))
            in_f = netcdf_file(fname, 'r')
        except IOError:
            print("!!! Missing entire day of forcing !!!\nRepeating the previous day.")
            date_file -= np.timedelta64(1, 'D')
            fname = data_root + to_datetime(date_file).strftime("%Y%m%d") + ".nc"
            fname = join(data_path, fname)
            xc, yc = get_COSMO_grid(fname)
            ind = np.where((xc>=xinfo[0]-15e3) & (xc<=xinfo[1]+15e3) & 
                           (yc>=yinfo[0]-15e3) & (yc<=yinfo[1]+15e3))
            in_f = netcdf_file(fname, 'r')
            missing_day = True
        print("date: %s\nfile: %s\n" % (today, fname))

        nc_time = get_time(in_f)
        if missing_day: #modified by Fazel
            req_time = np.arange(today - np.timedelta64(1, 'D'),
                                 today - np.timedelta64(1, 'D') + np.timedelta64(1, 'D'),
                                 np.timedelta64(1, 'h'))
            missing_day = False
        else:
            req_time = np.arange(today,
                                 today + np.timedelta64(1, 'D'),
                                 np.timedelta64(1, 'h'))
        # read and interpolate data
        extra = ""
        try:
            in_f.variables["U"]
        except KeyError:
            extra = "_10M"
        dims = len(in_f.variables["U"+extra].shape)
        if  dims == 4:
            u = in_f.variables["U"+extra][:, 0, ind[0], ind[1]]
            v = in_f.variables["V"+extra][:, 0, ind[0], ind[1]]
        elif  dims == 3:
            u = in_f.variables["U"+extra][:, ind[0], ind[1]]
            v = in_f.variables["V"+extra][:, ind[0], ind[1]]
        n_miss = 0
        for hh in req_time:
            try:
                index = np.where(nc_time == hh)[0][0]
                du = griddata((xc[ind[0], ind[1]], yc[ind[0], ind[1]]),
                               u[index, ...],
                               (xM.ravel(), yM.ravel()),
                               fill_value=np.nan, method="linear")
                dv = griddata((xc[ind[0], ind[1]], yc[ind[0], ind[1]]),
                               v[index, ...],
                               (xM.ravel(), yM.ravel()),
                               fill_value=np.nan, method="linear")
                bad = np.isnan(du)
                # if there are missing points, we fill them with
                # nearest neighbour values
                if np.any(bad > 0):
                    du[bad] = griddata((xc[ind[0], ind[1]],
                                        yc[ind[0], ind[1]]),
                                        u[index, ...],
                                        (xM.ravel()[bad],
                                         yM.ravel()[bad]),
                                        method="nearest")
                    dv[bad] = griddata((xc[ind[0], ind[1]],
                                        yc[ind[0], ind[1]]),
                                        v[index, ...],
                                        (xM.ravel()[bad],
                                         yM.ravel()[bad]),
                                        method="nearest")
                du = np.reshape(du, xM.shape)
                dv = np.reshape(dv, xM.shape)
                du, dv = rotate(du, dv, cM, sM)
                # spin up slowly the wind forcing
                if hh < (date_start + np.timedelta64(spin_up, 'D')):
                     coeff = (hh - date_start).astype("timedelta64[h]").astype(float) \
                             / (spin_up * 24.0)
                     du *= coeff
                     dv *= coeff
                speed = np.sqrt(du*du + dv*dv)
            except IndexError:
                print("Warning: missing data in file!")
                # we do not need to do anything, we will just be writing the
                # last out_data array we computed
                n_miss += 1
            # write to files
            for n,f in fls.items():
                if n == "uwind":
                    out_data = du
                elif n == "vwind":
                    out_data = dv
                elif n == "wspeed":
                    out_data = speed
                for kk in range(xM.shape[1]):
                    out_data[:,kk].tofile(f)
        if n_miss > 0:
            print("Not all expected times are available in this file:\n %s"
                  % nc_time)
            if n_miss == 24:
                raise ValueError("Nothing in this file")
        
        in_f.close()

        today += np.timedelta64(1, 'D')

    # close files
    for f in fls.values():
        f.close()

In [4]:
def cosmo2delft3d_uv_with_missing(data_path, data_root, start, end,
                  xinfo, yinfo,output_path, spin_up=0): # modified by Fazel

    spin_up = int(spin_up)

    # Time range to be extracted
    date_start = np.datetime64(start)
    date_end = np.datetime64(end)

    # Get grid info
    fname = data_root + to_datetime(date_start).strftime("%Y%m%d") + ".nc"
    fname = join(data_path,fname)

    # generate output grid
    (xM, yM), (cM, sM) = get_MITgcm_grid(mitgcm_grid_file)
    print(xM)

    # open the output files
    fls = {}
    for e in name:
        out_name = output_path+"%s_%s_%s.bin" % \
                   (e, start.replace('-',''), end.replace('-',''))
        fls[e] = open(out_name, "ab")

    # loop over COSMO-2 data
    today = date_start
    missing_day = False
    while today<=date_end:
#         # This is to follow a change in the naming convention
#         # so stupid one cannot believe
#         if today <= np.datetime64("2015-05-10"):
#             date_file = today
#         else:
#             date_file = today + np.timedelta64(1, 'D')

        date_file = today
        if ((date_file == np.datetime64('2021-09-03')) | (date_file == np.datetime64('2021-09-04'))):
            fname = 'cosmo-1e_ana_eawag_ens_' + to_datetime(date_file).strftime("%Y%m%d") + ".nc"
            fname = join('G:/ALPLakes/COSMO/cosmo_missing_dates_2021/', fname)
        else: 
            fname = data_root + to_datetime(date_file).strftime("%Y%m%d") + ".nc"
            fname = join(data_path, fname)
        try:
            xc, yc = get_COSMO_grid(fname)
            ind = np.where((xc>=xinfo[0]-15e3) & (xc<=xinfo[1]+15e3) & 
                           (yc>=yinfo[0]-15e3) & (yc<=yinfo[1]+15e3))
            in_f = netcdf_file(fname, 'r')
        except IOError:
            print("!!! Missing entire day of forcing !!!\nRepeating the previous day.")
            date_file -= np.timedelta64(1, 'D')
            fname = data_root + to_datetime(date_file).strftime("%Y%m%d") + ".nc"
            fname = join(data_path, fname)
            xc, yc = get_COSMO_grid(fname)
            ind = np.where((xc>=xinfo[0]-15e3) & (xc<=xinfo[1]+15e3) & 
                           (yc>=yinfo[0]-15e3) & (yc<=yinfo[1]+15e3))
            in_f = netcdf_file(fname, 'r')
            missing_day = True
        print("date: %s\nfile: %s\n" % (today, fname))

        nc_time = get_time(in_f)
        if missing_day: #modified by Fazel
            req_time = np.arange(today - np.timedelta64(1, 'D'),
                                 today - np.timedelta64(1, 'D') + np.timedelta64(1, 'D'),
                                 np.timedelta64(1, 'h'))
            missing_day = False
        else:
            req_time = np.arange(today,
                                 today + np.timedelta64(1, 'D'),
                                 np.timedelta64(1, 'h'))
        # read and interpolate data
        extra = ""
        
        if ((date_file == np.datetime64('2021-09-03')) | (date_file == np.datetime64('2021-09-04'))):
            u = in_f.variables["U_10M"][:, :, ind[0], ind[1]].mean(axis=1)
            v = in_f.variables["V_10M"][:, :, ind[0], ind[1]].mean(axis=1)
        else:
            try:
                in_f.variables["U"]
            except KeyError:
                extra = "_10M"
            dims = len(in_f.variables["U"+extra].shape)
            if  dims == 4:
                u = in_f.variables["U"+extra][:, 0, ind[0], ind[1]]
                v = in_f.variables["V"+extra][:, 0, ind[0], ind[1]]
            elif  dims == 3:
                u = in_f.variables["U"+extra][:, ind[0], ind[1]]
                v = in_f.variables["V"+extra][:, ind[0], ind[1]]
        
        
        n_miss = 0
        
        for hh in req_time:
            try:
                index = np.where(nc_time == hh)[0][0]
                du = griddata((xc[ind[0], ind[1]], yc[ind[0], ind[1]]),
                               u[index, ...],
                               (xM.ravel(), yM.ravel()),
                               fill_value=np.nan, method="linear")
                dv = griddata((xc[ind[0], ind[1]], yc[ind[0], ind[1]]),
                               v[index, ...],
                               (xM.ravel(), yM.ravel()),
                               fill_value=np.nan, method="linear")
                bad = np.isnan(du)
                # if there are missing points, we fill them with
                # nearest neighbour values
                if np.any(bad > 0):
                    du[bad] = griddata((xc[ind[0], ind[1]],
                                        yc[ind[0], ind[1]]),
                                        u[index, ...],
                                        (xM.ravel()[bad],
                                         yM.ravel()[bad]),
                                        method="nearest")
                    dv[bad] = griddata((xc[ind[0], ind[1]],
                                        yc[ind[0], ind[1]]),
                                        v[index, ...],
                                        (xM.ravel()[bad],
                                         yM.ravel()[bad]),
                                        method="nearest")
                du = np.reshape(du, xM.shape)
                dv = np.reshape(dv, xM.shape)
                du, dv = rotate(du, dv, cM, sM)
                # spin up slowly the wind forcing
                if hh < (date_start + np.timedelta64(spin_up, 'D')):
                     coeff = (hh - date_start).astype("timedelta64[h]").astype(float) \
                             / (spin_up * 24.0)
                     du *= coeff
                     dv *= coeff
                speed = np.sqrt(du*du + dv*dv)
            except IndexError:
                print("Warning: missing data in file!")
                # we do not need to do anything, we will just be writing the
                # last out_data array we computed
                n_miss += 1
            # write to files
            for n,f in fls.items():
                if n == "uwind":
                    out_data = du
                elif n == "vwind":
                    out_data = dv
                elif n == "wspeed":
                    out_data = speed
                for kk in range(xM.shape[1]):
                    out_data[:,kk].tofile(f)
#                     print(out_data.mean())
        if n_miss > 0:
            print("Not all expected times are available in this file:\n %s"
                  % nc_time)
            if n_miss == 24:
                raise ValueError("Nothing in this file")
        
        in_f.close()

        today += np.timedelta64(1, 'D')

    # close files
    for f in fls.values():
        f.close()

In [5]:
# cosmo2delft3d_uv('F:/COSMO/2021/','cosmo2_epfl_lakes_','2021-07-26','2021-07-28',[500000.0,563000.0,1000.0],[116500.0,138700.0,1000.0],'F:/Datalakes50_diag/binary_data/')
# cosmo2delft3d_uv_with_missing('G:/ALPLakes/COSMO/2021/','cosmo2_epfl_lakes_','2021-07-26','2021-09-10',[500000.0,563000.0,1000.0],[116500.0,138700.0,1000.0],'G:/ALPLakes/MITgcm/grid50_64cores_secchi_updated/binary_data/')
# cosmo2delft3d_uv_with_missing('H:/COSMO/2022/','cosmo2_epfl_lakes_','2022-08-01','2022-10-04',[500000.0,563000.0,1000.0],[116500.0,138700.0,1000.0],'F:/MITgcm_Dave/binary_data/')
cosmo2delft3d_uv_with_missing('F:/COSMO/2020/','cosmo2_epfl_lakes_','2020-04-15','2020-04-24',[506000.0,517000.0,500],[161000.0,171000.0,500],'G:/ALPLakes/D3D/lac_de_joux/LDJ_10days_2020/mitgcm_format/')

[[508547.       508626.3125   508719.78125  ... 515909.15625
  516039.3125   516169.46875 ]
 [508526.46875  508593.625    508682.5625   ... 515855.4375
  515979.34375  516103.25    ]
 [508475.984375 508560.9375   508645.34375  ... 515801.71875
  515919.375    516037.03125 ]
 ...
 [507944.796875 508005.25     508106.625    ... 514905.4375
  514935.71875  514966.      ]
 [507910.234375 507972.5625   508079.46875  ... 514862.53125
  514890.375    514918.21875 ]
 [507877.484375 507939.875    508052.3125   ... 514819.625
  514845.03125  514870.4375  ]]
date: 2020-04-15
file: F:/COSMO/2020/cosmo2_epfl_lakes_20200415.nc

date: 2020-04-16
file: F:/COSMO/2020/cosmo2_epfl_lakes_20200416.nc

date: 2020-04-17
file: F:/COSMO/2020/cosmo2_epfl_lakes_20200417.nc

date: 2020-04-18
file: F:/COSMO/2020/cosmo2_epfl_lakes_20200418.nc

date: 2020-04-19
file: F:/COSMO/2020/cosmo2_epfl_lakes_20200419.nc

date: 2020-04-20
file: F:/COSMO/2020/cosmo2_epfl_lakes_20200420.nc

date: 2020-04-21
file: F:/COSMO/2020/c